# Następna iteracja: confidence-aware selection

Ten notebook kontynuuje serię lokalnych iteracji po category-aware rerankingu i prototypach top-k.

Dotychczasowy najlepszy uczciwy wynik to pojedynczy `eegnet_top1_category_gate` (`SSIM ≈ 0.308`). Prototyp top-k osiągnął poziom VAE (`SSIM ≈ 0.286`), ale rozmywał detale i przegrywał z jednym kandydatem.

Pytanie tej iteracji: czy walidacja może nauczyć prostego selektora, kiedy wybrać nearest-neighbor, a kiedy prototyp?

## Hipoteza

Skoro dla różnych obrazów różne warianty bywają lepsze, być może prosta reguła wyboru metody poprawi wynik globalny.

Testujemy trzy selektory:

1. `global_validation_best` — wybierz jedną metodę, która ma najlepszy średni SSIM na walidacji.
2. `gate_category_validation_selector` — dla każdej kategorii przewidzianej przez EEGNet wybierz metodę, która była najlepsza na walidacji.
3. `oracle_per_image_upper_bound` — nieuczciwa górna granica: dla każdego obrazu testowego wybiera wariant o najlepszym SSIM. To służy tylko do diagnozy potencjału selektora.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'scripts').is_dir() else CWD.parent
OUTPUT_DIR = PROJECT_ROOT / 'wyniki colab' / 'confidence_aware_selection_mole_local'
SCRIPT = PROJECT_ROOT / 'scripts' / 'run_confidence_aware_selection.py'

paths = {
    'script': SCRIPT,
    'category-aware results': PROJECT_ROOT / 'wyniki colab' / 'category_aware_reranking_mole_local',
    'prototype results': PROJECT_ROOT / 'wyniki colab' / 'category_prototype_reconstruction_mole_local',
    'unclip embeddings': PROJECT_ROOT / 'image_embeddings_unclip_participant_image_mole_no_abc_local_20260627',
}
for name, path in paths.items():
    print(f'{name}:', path, 'OK' if path.exists() else 'BRAK')

In [ ]:
# Uruchom lokalny selektor.
import subprocess, sys

cmd = [
    sys.executable, str(SCRIPT),
    '--project-root', str(PROJECT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Wynik selektorów.
import json
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_DIR / 'confidence_selection_summary.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(OUTPUT_DIR / 'selector_comparison.csv')
display(comparison)

print('Global best na walidacji:', summary['global_validation_best_method'])
print('Metody per gate category:')
display(pd.DataFrame([summary['gate_category_methods']]).T.rename(columns={0: 'method'}))

In [ ]:
# Podgląd gridów.
from IPython.display import Image as IPImage, Markdown, display

for path in sorted((OUTPUT_DIR / 'grids').glob('*.jpg')):
    display(Markdown(f'### {path.name}'))
    display(IPImage(filename=str(path)))

## Wpis historyczny — wynik lokalny z 2026-06-27

Selektor globalny wybrał `nn_eegnet_top1_gate`, czyli tę samą metodę, która była dotychczas najlepsza. Na teście zachował `SSIM = 0.308`.

Selektor per przewidziana kategoria pogorszył wynik do `SSIM = 0.292`. Użył prototypu w 20/44 przypadkach, ale prototyp często rozmywał szczegóły.

Najważniejszy wynik diagnostyczny to oracle per-obraz wśród tych samych nie-oracle kandydatów: `SSIM = 0.386`. To oznacza, że zestaw kandydatów ma realny zapas jakości, ale prosta reguła po kategorii nie potrafi go wydobyć.

Wniosek dla następnej iteracji: jeśli chcemy adaptacyjnie wybierać wariant rekonstrukcji, potrzebujemy lepszego predyktora zaufania/confidence niż sama przewidziana kategoria. Sensowne cechy to margin score'ów, entropia EEGNet, zgodność top-k kategorii, hubness score kandydata i stabilność między powtórzeniami EEG.